In [0]:
%pip install ax-platform
%pip install xgboost
%restart_python

In [0]:
import importlib
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, PredictionErrorDisplay

from xgboost import XGBRegressor, plot_importance

from ax.service.managed_loop import optimize
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig

In [0]:
df_desktop = spark.sql('''
SELECT * 
FROM workspace.`site-speed-recommendation`.combined
WHERE performance_score is not null and strategy="desktop"
''').toPandas()

df_mobile = spark.sql('''
SELECT * 
FROM workspace.`site-speed-recommendation`.combined
WHERE performance_score is not null and strategy="mobile"
''').toPandas()

In [0]:
df_desktop_copy = df_desktop.copy()
df_mobile_copy = df_mobile.copy()

high_missing_col = ['mainthread_garbageCollection','EXPERIMENTAL_TIME_TO_FIRST_BYTE','INTERACTION_TO_NEXT_PAINT']
df_desktop_copy = df_desktop_copy.drop(columns=high_missing_col)
df_mobile_copy = df_mobile_copy.drop(columns=high_missing_col)
df_desktop_copy=df_desktop_copy.dropna()
df_mobile_copy=df_mobile_copy.dropna()

y_desktop = df_desktop_copy['largest-contentful-paint']
y_mobile = df_mobile_copy['largest-contentful-paint']

drop_cols = ['domain', 'url', 'performance_score', 'largest-contentful-paint','cumulative-layout-shift','first-contentful-paint','total-blocking-time','speed-index', 'interactive', 'strategy']
df_desktop_copy = df_desktop_copy.drop(columns=drop_cols)
df_mobile_copy = df_mobile_copy.drop(columns=drop_cols)

In [0]:
y_mobile.describe()
y_mobile.quantile([0.75,0.85,0.9,0.95])

In [0]:
y_mobile_upper = y_mobile.quantile(0.9)
y_mobile = y_mobile.clip(0, y_mobile_upper)

In [0]:
# X_desktop_encoded = pd.get_dummies(
#     df_desktop_copy,
#     columns=["strategy"],
#     drop_first=False
# )
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(df_desktop_copy, y_desktop, test_size=0.2, random_state=42)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(df_mobile_copy, y_mobile, test_size=0.2, random_state=42)

In [0]:
# client_desktop = Client()
client_mobile = Client()

In [0]:
# client_desktop.configure_experiment(
#     parameters=[
#         RangeParameterConfig(
#             name="max_depth",
#             parameter_type="int",
#             bounds=(3, 6),
#         ),
#         RangeParameterConfig(
#             name="learning_rate",
#             parameter_type="float",
#             bounds=(0.01, 0.3),
#         ),
#         RangeParameterConfig(
#             name="subsample",
#             parameter_type="float",
#             bounds=(0.5, 1.0),
#         ),
#         RangeParameterConfig(
#             name="colsample_bytree",
#             parameter_type="float",
#             bounds=(0.5, 1.0),
#         ),
#         RangeParameterConfig(
#             name="n_estimators",
#             parameter_type="int",
#             bounds=(100, 1000),
#         ),
#         RangeParameterConfig(
#             name="min_child_weight",
#             parameter_type="int",
#             bounds=(1, 20),
#         ),
#         RangeParameterConfig(
#             name="gamma",
#             parameter_type="float",
#             bounds=(0, 1.0),
#         ),
#         RangeParameterConfig(
#             name="reg_alpha",
#             parameter_type="float",
#             bounds=(0.0, 5.0),
#         ),
#         RangeParameterConfig(
#             name="reg_lambda",
#             parameter_type="float",
#             bounds=(0.01, 100.0),
#             scaling="log"
#         ),
#     ]
# )
# client_desktop.configure_optimization(objective="-mae")

client_mobile.configure_experiment(
    parameters=[
        RangeParameterConfig(
            name="max_depth",
            parameter_type="int",
            bounds=(3, 5),
        ),
        RangeParameterConfig(
            name="learning_rate",
            parameter_type="float",
            bounds=(0.01, 0.3),
        ),
        RangeParameterConfig(
            name="subsample",
            parameter_type="float",
            bounds=(0.5, 1.0),
        ),
        RangeParameterConfig(
            name="colsample_bytree",
            parameter_type="float",
            bounds=(0.5, 1.0),
        ),
        RangeParameterConfig(
            name="n_estimators",
            parameter_type="int",
            bounds=(100, 1000),
        ),
        RangeParameterConfig(
            name="min_child_weight",
            parameter_type="int",
            bounds=(1, 20),
        ),
        RangeParameterConfig(
            name="gamma",
            parameter_type="float",
            bounds=(0, 1.0),
        ),
        RangeParameterConfig(
            name="reg_alpha",
            parameter_type="float",
            bounds=(0.0, 5.0),
        ),
        RangeParameterConfig(
            name="reg_lambda",
            parameter_type="float",
            bounds=(0.01, 100.0),
            scaling="log"
        ),
    ]
)
client_mobile.configure_optimization(objective="-mae")

In [0]:
def evaluate_xgb(parameters, X_train, y_train):
    base_model = XGBRegressor(
        max_depth=int(parameters["max_depth"]),
        learning_rate=parameters["learning_rate"],
        n_estimators=int(parameters["n_estimators"]),
        subsample=parameters["subsample"],
        colsample_bytree=parameters["colsample_bytree"],
        min_child_weight=parameters["min_child_weight"],
        reg_alpha=parameters["reg_alpha"],
        reg_lambda=parameters["reg_lambda"],
        gamma=parameters["gamma"],
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )

    model = TransformedTargetRegressor(
        regressor=base_model,
        func=np.log1p,
        inverse_func=np.expm1
    )

    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1,
    )

    mae = -scores.mean()

    return {"mae": mae}

In [0]:
# for i in range(40):
#     trails = client_desktop.get_next_trials(max_trials=1)

#     for trial_index, parameters in trails.items():
#         result = evaluate_xgb(parameters, X_train_d, y_train_d)

#         client_desktop.complete_trial(
#             trial_index = trial_index,
#             raw_data = result,
#         )

#         print(f"Trial {trial_index}: MAE = {result["mae"]:.4f}")
#         print(parameters)

for i in range(40):
    trails = client_mobile.get_next_trials(max_trials=1)

    for trial_index, parameters in trails.items():
        result = evaluate_xgb(parameters, X_train_m, y_train_m)

        client_mobile.complete_trial(
            trial_index = trial_index,
            raw_data = result,
        )

        print(f"Trial {trial_index}: MAE = {result["mae"]:.4f}")
        print(parameters)

In [0]:
# best_parameters_desktop = client_desktop.get_best_parameterization()
# print(best_parameters_desktop)

best_parameters_mobile = client_mobile.get_best_parameterization()
print(best_parameters_mobile)

In [0]:
{'max_depth': 5, 'learning_rate': 0.10335942050506403, 'subsample': 0.8078683281580821, 'colsample_bytree': 1.0, 'n_estimators': 940, 'min_child_weight': 1, 'gamma': 0.0, 'reg_alpha': 1.5837724646040927, 'reg_lambda': 3.4089347648449544}
xgb = XGBRegressor(
    max_depth=5,
    learning_rate=0.10335942050506403,
    subsample=0.8078683281580821,
    colsample_bytree=1,
    n_estimators=940,
    min_child_weight=1,
    gamma=0.0,
    reg_alpha=1.5837724646040927,
    reg_lambda=3.4089347648449544,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

model = TransformedTargetRegressor(
    regressor=xgb,
    func=np.log1p,
    inverse_func=np.expm1
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model,
    X_train_m,
    y_train_m,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

print("XGBoost CV MAE scores:", -scores)
print("Mean CV MAE:", -scores.mean())
print("CV MAE std:", scores.std())

model.fit(X_train_m, y_train_m)

train_pred = model.predict(X_train_m)
test_pred = model.predict(X_test_m)

train_mae = mean_absolute_error(y_train_m, train_pred)
test_mae = mean_absolute_error(y_test_m, test_pred)

rmse = np.sqrt(mean_squared_error(y_test_m, test_pred))
r2 = r2_score(y_test_m, test_pred)

display = PredictionErrorDisplay(y_true=y_test_m, y_pred=test_pred)
display.plot()

print(f"Train MAE: {train_mae}")
print(f"Test MAE: {test_mae}")
print(f"Test R-squared: {r2}")
print(f"Test RMSE: {rmse}")

In [0]:
%skip
{'max_depth': 5, 'learning_rate': 0.10335942050506403, 'subsample': 0.8078683281580821, 'colsample_bytree': 1.0, 'n_estimators': 940, 'min_child_weight': 1, 'gamma': 0.0, 'reg_alpha': 1.5837724646040927, 'reg_lambda': 3.4089347648449544}
XGBoost CV MAE scores: [1852.53563154 1813.0029862  1890.18819057 1944.09978925 1890.22858403]
Mean CV MAE: 1878.0110363212602
CV MAE std: 43.681386541047935
Train MAE: 1157.4375183166617
Test MAE: 1791.169130728298
Test R-squared: 0.8643081439645257
Test RMSE: 2774.0925964647663

In [0]:
feature_columns = X_train.columns.tolist()
feature_columns

In [0]:
fig, ax = plt.subplots(figsize=(8, 5))

plot_importance(
    model.regressor_,
    ax=ax,
    max_num_features=15,
    importance_type="gain"
)

plt.show()

In [0]:
importances = pd.Series(
    model.regressor_.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importances)

In [0]:
# %skip
# model.regressor_.save_model("/Volumes/workspace/site-speed-recommendation/models/0001.bin")
joblib.dump(model, "/Volumes/workspace/site-speed-recommendation/models/lcp_model.joblib")